In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import requests
import os
import dotenv
dotenv.load_dotenv()

True

In [3]:
data = PyPDFLoader(file_path='/Users/dipakkhade/projects/AI-ML-100x/week-09 RAG from the Ground Up - Part 1 /docs/Mahabharata.pdf')
docs = data.load()

In [4]:
txt_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=400 
)

document_chunks = txt_splitter.split_documents(documents=docs)

texts = [doc.page_content for doc in document_chunks]

In [5]:
response = requests.post(
  "https://openrouter.ai/api/v1/embeddings",
  headers={
    "Authorization": f"Bearer {os.environ.get('OPENROUTER_API_KEY')}",
    "Content-Type": "application/json",
  },
  json={
    "model": "openai/text-embedding-3-small",
    "input": "The quick brown fox jumps over the lazy dog"
  }
)
data = response.json()
embedding = data["data"][0]["embedding"]
print(f"Embedding dimension: {len(embedding)}")

Embedding dimension: 1536


In [6]:
chroma_client = chromadb.HttpClient(host='localhost', port=8000)

mahabharat_collection = chroma_client.create_collection('mahabharat2')

mahabharat_collection.add(
    ids=['1'],
    embeddings=embedding
)

ChromaError: Collection [mahabharat2] already exists

In [7]:
collections = chroma_client.list_collections()
print(collections)

[Collection(name=mahabharat2), Collection(name=mahabharat)]


In [8]:
mahabharat2_collection = chroma_client.get_collection(name="mahabharat2")
results = mahabharat2_collection.query(
    query_texts=["who is Arjun"], # The query text
    n_results=2                                # Number of results to return
)

print(results)


/Users/dipakkhade/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:14<00:00, 5.68MiB/s]


InvalidArgumentError: Collection expecting embedding with dimension of 1536, got 384